In [16]:
import pandas as pd
import numpy as np

In [17]:
val   = pd.read_csv("data/val.csv")
test  = pd.read_csv("data/test.csv")
train = pd.read_csv("data/train.csv")

In [18]:
origin_date = pd.to_datetime(train["Date"]).min()

In [19]:
def encode_dataset(df):
    df_encoded = df.copy()

    # DATE FEATURES
    df_encoded["Date"] = pd.to_datetime(df_encoded["Date"])

    df_encoded["time_idx"] = (
        df_encoded["Date"] - origin_date
    ).dt.days

    month = df_encoded["Date"].dt.month
    dayofweek = df_encoded["Date"].dt.dayofweek

    df_encoded["is_weekend"] = (dayofweek >= 5).astype(int)

    df_encoded["month_sin"] = np.sin(2 * np.pi * month / 12)
    df_encoded["month_cos"] = np.cos(2 * np.pi * month / 12)

    df_encoded["dow_sin"] = np.sin(2 * np.pi * dayofweek / 7)
    df_encoded["dow_cos"] = np.cos(2 * np.pi * dayofweek / 7)

    # FREQUENCY ENCODING (uses TRAIN maps)
    df_encoded["store_freq"] = df_encoded["Store ID"].map(store_freq_map).fillna(0)
    df_encoded["product_freq"] = df_encoded["Product ID"].map(product_freq_map).fillna(0)

    store_product_key = (
        df_encoded["Store ID"].astype(str)
        + "_" +
        df_encoded["Product ID"].astype(str)
    )

    df_encoded["store_product_freq"] = (
        store_product_key.map(store_product_freq_map).fillna(0)
    )
    

    # DROP RAW
    df_encoded.drop(columns=["Date", "Store ID", "Product ID"], inplace=True)

    # ONE HOT
    low_card_cols = df_encoded.select_dtypes(include="object").columns

    df_encoded = pd.get_dummies(df_encoded, columns=low_card_cols, dtype=int)

    return df_encoded

In [20]:
# MUST run this first



store_freq_map = train["Store ID"].value_counts().to_dict()
product_freq_map = train["Product ID"].value_counts().to_dict()

store_product_freq_map = (
    train["Store ID"].astype(str) + "_" +
    train["Product ID"].astype(str)
).value_counts().to_dict()

In [21]:
def transform_for_prediction(df):
    encoded = encode_dataset(df)

    # 🔥 THIS is the key step
    encoded = encoded.reindex(columns=train_columns, fill_value=0)

    return encoded

In [24]:
train_encoded = pd.read_csv("data/transformed_train.csv")

train_columns = train_encoded.columns.tolist()

In [25]:
val_encoded = transform_for_prediction(val)
test_encoded = transform_for_prediction(test)

In [26]:
val_encoded.to_csv("transformed_val.csv", index=False)
test_encoded.to_csv("transformed_test.csv", index=False)